In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

df = pd.read_csv("../data/SMSSpamCollection", sep="\t", names=["label", "message"])
X_train, X_test, y_train, y_test = train_test_split(
    df["message"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

vectorizer = TfidfVectorizer(stop_words="english")
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [6]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)
pred = model.predict(X_test_tfidf)

print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred, labels=["spam", "ham"]))

              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       966
        spam       1.00      0.76      0.86       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.97      1115

[[113  36]
 [  0 966]]


In [7]:
results = pd.DataFrame({"message": X_test, "actual": y_test, "predicted": pred})
missed = results[(results["actual"] == "spam") & (results["predicted"] == "ham")]

for m in missed["message"].head(5):
    print("-", m)

- You are now unsubscribed all services. Get tons of sexy babes or hunks straight to your phone! go to http://gotbabes.co.uk. No subscriptions.
- FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, £1.50 to rcv
- Dear Voucher Holder 2 claim your 1st class airport lounge passes when using Your holiday voucher call 08704439680. When booking quote 1st class x 2
- Hi if ur lookin 4 saucy daytime fun wiv busty married woman Am free all next week Chat now 2 sort time 09099726429 JANINExx Calls£1/minMobsmoreLKPOBOX177HP51FL
- ringtoneking 84484


In [8]:
model_bal = LogisticRegression(max_iter=1000, class_weight="balanced")
model_bal.fit(X_train_tfidf, y_train)
pred_bal = model_bal.predict(X_test_tfidf)

print(classification_report(y_test, pred_bal))
print(confusion_matrix(y_test, pred_bal, labels=["spam", "ham"]))

              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       966
        spam       0.91      0.91      0.91       149

    accuracy                           0.98      1115
   macro avg       0.95      0.95      0.95      1115
weighted avg       0.98      0.98      0.98      1115

[[136  13]
 [ 13 953]]


In [9]:
import numpy as np
from sklearn.metrics import precision_score, recall_score

proba = model.predict_proba(X_test_tfidf)[:, 1]   # chance of spam

for t in [0.2, 0.3, 0.4, 0.5]:
    pred_t = np.where(proba >= t, "spam", "ham")
    p = precision_score(y_test, pred_t, pos_label="spam")
    r = recall_score(y_test, pred_t, pos_label="spam")
    print(f"threshold {t}: precision {p:.2f}, recall {r:.2f}")

threshold 0.2: precision 0.90, recall 0.93
threshold 0.3: precision 0.96, recall 0.87
threshold 0.4: precision 0.99, recall 0.84
threshold 0.5: precision 1.00, recall 0.76
